# 04C_NLP_Skill_Extraction_SES_Pipeline

KeyBERT/Skill Extraction → Classification → SES Feature Dataset

In [4]:
!pip install keybert sentence-transformers spacy tqdm -q

In [5]:
import pandas as pd
import numpy as np
from keybert import KeyBERT
from tqdm import tqdm
from collections import Counter

tqdm.pandas()
kw_model = KeyBERT()

C:\Users\Saanvi\anaconda3\envs\COURSE\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Saanvi\anaconda3\envs\COURSE\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Saanvi\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In 

In [6]:
postings = pd.read_csv('postings.csv', low_memory=False)
linkedin = pd.read_csv('linkedin_job_postings.csv', low_memory=False)
stack = pd.read_csv('survey_results_public.csv', low_memory=False)

In [7]:
postings['text'] = (
    postings['title'].fillna('') + ' ' +
    postings['description'].fillna('') + ' ' +
    postings['skills_desc'].fillna('')
)

In [8]:
def extract_candidates(text):
    try:
        kws = kw_model.extract_keywords(
            str(text)[:4000],
            keyphrase_ngram_range=(1,3),
            stop_words='english',
            top_n=20
        )
        return [x[0].lower() for x in kws]
    except:
        return []

In [ ]:
sample = postings.copy()
sample['candidate_skills'] = sample['text'].progress_apply(extract_candidates)

  0%|                                                                           | 77/123849 [00:48<28:38:32,  1.20it/s]

In [ ]:
TECH = ['python','java','sql','javascript','typescript','c++','c#','go','rust']
CLOUD = ['aws','azure','gcp']
DATABASE = ['mysql','postgresql','mongodb','redis','oracle']
FRAMEWORK = ['react','node','django','flask','fastapi','spring','angular','vue']
TOOL = ['docker','kubernetes','git','tableau','power bi','spark','airflow']
SOFT = ['communication','leadership','teamwork','problem solving','critical thinking','adaptability']

In [ ]:
def classify_skill(skill):
    s = skill.lower()
    if any(x in s for x in TECH):
        return 'Technical'
    if any(x in s for x in CLOUD):
        return 'Cloud'
    if any(x in s for x in DATABASE):
        return 'Database'
    if any(x in s for x in FRAMEWORK):
        return 'Framework'
    if any(x in s for x in TOOL):
        return 'Tool'
    if any(x in s for x in SOFT):
        return 'Soft Skill'
    return 'Unclassified'

In [ ]:
rows=[]
for _,r in sample.iterrows():
    sal = r.get('normalized_salary', np.nan)
    for skill in r['candidate_skills']:
        rows.append([skill, classify_skill(skill), sal])

skills_df = pd.DataFrame(rows, columns=['skill','category','salary'])

In [ ]:
demand = skills_df['skill'].value_counts().reset_index()
demand.columns=['skill','linkedin_demand']

salary = skills_df.groupby('skill')['salary'].median().reset_index()
salary.columns=['skill','salary_premium']

In [ ]:
master = demand.merge(salary,on='skill',how='left')
master = master.merge(skills_df[['skill','category']].drop_duplicates(),on='skill',how='left')
master.head(20)

In [ ]:
master.to_csv('nlp_extracted_skill_master.csv',index=False)
print(master.shape)